# Limpieza y preprocesamiento de datos: BD Nacionalidad

Este notebook documenta el proceso de limpieza y preparación del dataset
**BD_Nacionalidad**, que registra el flujo de turistas extranjeros que ingresaron
a México por vía aérea, clasificados por aeropuerto, país de nacionalidad,
sexo y periodo (2012–2026).

A diferencia del dataset de Residencia, esta fuente permite identificar
la nacionalidad real del visitante (no su país de residencia habitual),
lo que enriquece el análisis de origen de los flujos turísticos.

El producto final de este notebook es **`df_nac_modelo`**: un dataset limpio,
normalizado y filtrado, listo para integrarse al pipeline de modelado predictivo.

In [ ]:
!pip install openpyxl --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("Librerías importadas y entorno configurado.")

Librerías importadas y entorno configurado.


## 1. Carga del dataset

El dataset se descarga directamente desde el repositorio del proyecto en GitHub
en formato `.xlsx`. Se imprime la dimensión resultante para confirmar
una lectura exitosa.

In [ ]:
url_nacionalidad = "https://github.com/Montiel-Oscar/ciencia_de_datos_2026_2/raw/283d4ed6068f70af81e280f88ceb73cc09c7432a/proyecto_borrador/datasets/BD_Nacionalidad.xlsx"

print("Descargando y leyendo el archivo, esto puede tardar unos segundos...")
df_nac = pd.read_excel(url_nacionalidad, engine="openpyxl")

print(f"Dataset cargado exitosamente.")
print(f"Forma inicial: {df_nac.shape[0]:,} filas x {df_nac.shape[1]} columnas")
display(df_nac.head())

Descargando y leyendo el archivo, esto puede tardar unos segundos...
Dataset cargado exitosamente.
Forma inicial: 500,565 filas x 10 columnas


,Año,Fecha,MesNum,Mes,Aeropuerto,Origen,Pais,Región,Sexo,Valor
0,2012,2012/1/1,1,Enero,"Acapulco, Gro.",Nacionalidad,Alemania,Europa,Hombre,1
1,2012,2012/1/1,1,Enero,"Acapulco, Gro.",Nacionalidad,Alemania,Europa,Mujer,1
2,2012,2012/1/1,1,Enero,"Acapulco, Gro.",Nacionalidad,Australia,Oceanía,Hombre,6
3,2012,2012/1/1,1,Enero,"Acapulco, Gro.",Nacionalidad,Australia,Oceanía,Mujer,1
4,2012,2012/1/1,1,Enero,"Acapulco, Gro.",Nacionalidad,Austria,Europa,Hombre,1


## 2. Exploración general

Se genera un resumen técnico del dataset que incluye el tipo de dato de cada
columna, el conteo y porcentaje de valores nulos, y el número de valores únicos.
Adicionalmente se verifican las columnas `Origen` y `Sexo`, que en el dataset
de Residencia presentaron problemas: valor constante y categorías no clasificadas,
respectivamente.

In [ ]:
print("="*50)
print("RADIOGRAFÍA DEL DATASET: BD_NACIONALIDAD")
print("="*50)

info_nac = pd.DataFrame({
    'Tipo de Dato': df_nac.dtypes,
    'Valores Nulos': df_nac.isnull().sum(),
    '% Nulo': (df_nac.isnull().sum() / len(df_nac) * 100).round(2),
    'Valores Únicos': df_nac.nunique()
})

display(info_nac)

print("\n--- Verificaciones Rápidas ---")
if 'Origen' in df_nac.columns:
    print(f"Valores en 'Origen': {df_nac['Origen'].unique()}")
if 'Sexo' in df_nac.columns:
    print(f"Conteo de 'Sexo':\n{df_nac['Sexo'].value_counts()}")

RADIOGRAFÍA DEL DATASET: BD_NACIONALIDAD


,Tipo de Dato,Valores Nulos,% Nulo,Valores Únicos
Año,int64,0,0.0,15
Fecha,object,0,0.0,169
MesNum,int64,0,0.0,12
Mes,object,0,0.0,12
Aeropuerto,object,0,0.0,67
Origen,object,0,0.0,1
Pais,object,0,0.0,239
Región,object,0,0.0,10
Sexo,object,0,0.0,3
Valor,int64,0,0.0,10307



--- Verificaciones Rápidas ---
Valores en 'Origen': ['Nacionalidad']
Conteo de 'Sexo':
Sexo
Hombre           273633
Mujer            226763
No disponible       169
Name: count, dtype: int64


## 3. Limpieza de datos

Se aplican de forma consolidada las tres transformaciones estructurales
identificadas en la exploración:

1. **Eliminación de columnas redundantes**: `Origen` (valor único constante),
   `Fecha` (redundante con `Año` y `MesNum`) y `Mes` (redundante con `MesNum` numérico).
2. **Eliminación de registros sin clasificación de género**: se descartan los registros
   cuya columna `Sexo` contiene "No disponible", los cuales carecen también de
   información de región y no son imputables.
3. **Generación del dataset modelo**: se excluyen los años de pandemia (2020–2021)
   y el año en curso incompleto (2026), conservando el periodo 2012–2019 y 2022–2025
   para el entrenamiento del modelo predictivo.

In [ ]:
print("Iniciando limpieza estructural...")

cols_a_eliminar = ['Origen', 'Fecha', 'Mes']
df_nac.drop(columns=[col for col in cols_a_eliminar if col in df_nac.columns], inplace=True)
print(f"Columnas eliminadas: {cols_a_eliminar}")

filas_antes = len(df_nac)
df_nac = df_nac[df_nac['Sexo'] != 'No disponible'].copy()
print(f"Registros 'No disponible' eliminados: {filas_antes - len(df_nac)}")

df_nac_modelo = df_nac[~df_nac['Año'].isin([2020, 2021, 2026])].copy()
print(f"Filas restantes para el modelo (sin 2020, 2021, 2026): {len(df_nac_modelo):,}")

Iniciando limpieza estructural...
Columnas eliminadas: ['Origen', 'Fecha', 'Mes']
Registros 'No disponible' eliminados: 169
Filas restantes para el modelo (sin 2020, 2021, 2026): 445,547


## 4. Normalización de variables categóricas

Las columnas de texto (`Aeropuerto`, `Pais`, `Región`, `Sexo`) se homogenizan
mediante tres operaciones secuenciales: conversión a minúsculas, eliminación de
acentos y caracteres especiales mediante normalización Unicode (NFKD),
y supresión de espacios en blanco al inicio y al final de cada valor.
Esta estandarización garantiza la consistencia de las categorías durante el modelado.

In [ ]:
print("Iniciando normalización de texto...")

columnas_texto = ['Aeropuerto', 'Pais', 'Región', 'Sexo']

for col in columnas_texto:
    df_nac_modelo[col] = df_nac_modelo[col].str.lower()

    df_nac_modelo[col] = df_nac_modelo[col].str.normalize('NFKD')\
                                           .str.encode('ascii', errors='ignore')\
                                           .str.decode('utf-8')

    df_nac_modelo[col] = df_nac_modelo[col].str.strip()

print("Normalización completada exitosamente.")

Iniciando normalización de texto...
Normalización completada exitosamente.


## 5. Verificación del dataset final

Se presenta el estado definitivo del dataset modelo: dimensiones, columnas
resultantes y una muestra de los datos normalizados. Se verifica también
que los valores de la columna `Región` hayan sido correctamente estandarizados.

In [ ]:
print("="*50)
print("ESTADO FINAL DEL DATASET NACIONALIDAD (MODELO)")
print("="*50)
print(f"Forma final: {df_nac_modelo.shape[0]:,} filas x {df_nac_modelo.shape[1]} columnas")
print(f"Columnas actuales: {df_nac_modelo.columns.tolist()}")

print("\nMuestra de datos normalizados:")
display(df_nac_modelo.head())

print("\nValidación de normalización en 'Región':")
print(df_nac_modelo['Región'].unique())

ESTADO FINAL DEL DATASET NACIONALIDAD (MODELO)
Forma final: 445,547 filas x 7 columnas
Columnas actuales: ['Año', 'MesNum', 'Aeropuerto', 'Pais', 'Región', 'Sexo', 'Valor']

Muestra de datos normalizados:


,Año,MesNum,Aeropuerto,Pais,Región,Sexo,Valor
0,2012,1,"acapulco, gro.",alemania,europa,hombre,1
1,2012,1,"acapulco, gro.",alemania,europa,mujer,1
2,2012,1,"acapulco, gro.",australia,oceania,hombre,6
3,2012,1,"acapulco, gro.",australia,oceania,mujer,1
4,2012,1,"acapulco, gro.",austria,europa,hombre,1



Validación de normalización en 'Región':
['europa' 'oceania' 'islas del caribe' 'america del sur'
 'america del norte' 'asia' 'america central' 'africa' 'apatrida']


## 6. Exportación del dataset limpio

El dataset modelo, completamente limpio y normalizado, se exporta en formato CSV
para su integración con los demás datasets del proyecto en las etapas de
modelado y evaluación.

In [ ]:
# Celda 7: Exportar y descargar el dataset modelo de Nacionalidad
print("Guardando BD_Nacionalidad_modelo.csv...")
df_nac_modelo.to_csv('BD_Nacionalidad_modelo.csv', index=False)

from google.colab import files
files.download('BD_Nacionalidad_modelo.csv')

print("¡Archivo guardado y descargado exitosamente!")

Guardando BD_Nacionalidad_modelo.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

¡Archivo guardado y descargado exitosamente!
